## MDP setup for RL training
- States: Min-max scaled ohlcv for 7, 14, 24, 168, 1500 hours
- Actions: Either tack the coin or cash
- Rewards: if coin: $log{rand(L_{t+1}, H_{t+1}) \over C_{t}}$, else 0

In [1]:
import random
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_finance import candlestick_ohlc
import matplotlib.dates as mdates

from binance import Client

/opt/homebrew/Caskroom/miniforge/base/envs/coin/lib/python3.9/site-packages/mpl_finance.py:16: DeprecationWarning: 



    Please use `mplfinance` instead (no hyphen, no underscore).

    To install: `pip install --upgrade mplfinance` 

   For more information, see: https://pypi.org/project/mplfinance/


  __warnings.warn('\n\n  ================================================================='+


In [5]:
client = Client()
klines = np.array(client.get_historical_klines("ETHUSDT", Client.KLINE_INTERVAL_1HOUR, "1 Jan, 2021"))
sample = pd.DataFrame(klines.reshape(-1, 12), dtype=float, columns=['datetime',
                                                                   'open',
                                                                   'high',
                                                                   'low',
                                                                   'close',
                                                                   'volume',
                                                                   'close time',
                                                                   'quote asset volume, number of trades',
                                                                   'number of trades',
                                                                   'taker buy base asset volume',
                                                                   'taker buy quote asset volume',
                                                                   'ignore'])
sample['datetime'] = pd.to_datetime(sample['datetime'], unit='ms')
sample.set_index('datetime', inplace=True)
sample = sample[['open', 'high', 'low', 'close', 'volume']]

In [6]:
# make states
sample['ropen_7'] = (sample['open'] - sample['low'].rolling(7).min())/(sample['high'].rolling(7).max() - sample['low'].rolling(7).min())
sample['rhigh_7'] = (sample['high'] - sample['low'].rolling(7).min())/(sample['high'].rolling(7).max() - sample['low'].rolling(7).min())
sample['rlow_7'] = (sample['low'] - sample['low'].rolling(7).min())/(sample['high'].rolling(7).max() - sample['low'].rolling(7).min())
sample['rclose_7'] = (sample['close'] - sample['low'].rolling(7).min())/(sample['high'].rolling(7).max() - sample['low'].rolling(7).min())
sample['rvolume_7'] = (sample['volume'] - sample['volume'].rolling(7).min())/(sample['volume'].rolling(7).max() - sample['volume'].rolling(7).min())

sample['ropen_14'] = (sample['open'] - sample['low'].rolling(14).min())/(sample['high'].rolling(14).max() - sample['low'].rolling(14).min())
sample['rhigh_14'] = (sample['high'] - sample['low'].rolling(14).min())/(sample['high'].rolling(14).max() - sample['low'].rolling(14).min())
sample['rlow_14'] = (sample['low'] - sample['low'].rolling(14).min())/(sample['high'].rolling(14).max() - sample['low'].rolling(14).min())
sample['rclose_14'] = (sample['close'] - sample['low'].rolling(14).min())/(sample['high'].rolling(14).max() - sample['low'].rolling(14).min())
sample['rvolume_14'] = (sample['volume'] - sample['volume'].rolling(14).min())/(sample['volume'].rolling(14).max() - sample['volume'].rolling(14).min())

sample['ropen_24'] = (sample['open'] - sample['low'].rolling(24).min())/(sample['high'].rolling(24).max() - sample['low'].rolling(24).min())
sample['rhigh_24'] = (sample['high'] - sample['low'].rolling(24).min())/(sample['high'].rolling(24).max() - sample['low'].rolling(24).min())
sample['rlow_24'] = (sample['low'] - sample['low'].rolling(24).min())/(sample['high'].rolling(24).max() - sample['low'].rolling(24).min())
sample['rclose_24'] = (sample['close'] - sample['low'].rolling(24).min())/(sample['high'].rolling(24).max() - sample['low'].rolling(24).min())
sample['rvolume_24'] = (sample['volume'] - sample['volume'].rolling(24).min())/(sample['volume'].rolling(24).max() - sample['volume'].rolling(24).min())

sample['ropen_168'] = (sample['open'] - sample['low'].rolling(168).min())/(sample['high'].rolling(168).max() - sample['low'].rolling(168).min())
sample['rhigh_168'] = (sample['high'] - sample['low'].rolling(168).min())/(sample['high'].rolling(168).max() - sample['low'].rolling(168).min())
sample['rlow_168'] = (sample['low'] - sample['low'].rolling(168).min())/(sample['high'].rolling(168).max() - sample['low'].rolling(168).min())
sample['rclose_168'] = (sample['close'] - sample['low'].rolling(168).min())/(sample['high'].rolling(168).max() - sample['low'].rolling(168).min())
sample['rvolume_168'] = (sample['volume'] - sample['volume'].rolling(168).min())/(sample['volume'].rolling(168).max() - sample['volume'].rolling(168).min())

sample['ropen_1500'] = (sample['open'] - sample['low'].rolling(1500).min())/(sample['high'].rolling(1500).max() - sample['low'].rolling(1500).min())
sample['rhigh_1500'] = (sample['high'] - sample['low'].rolling(1500).min())/(sample['high'].rolling(1500).max() - sample['low'].rolling(1500).min())
sample['rlow_1500'] = (sample['low'] - sample['low'].rolling(1500).min())/(sample['high'].rolling(1500).max() - sample['low'].rolling(1500).min())
sample['rclose_1500'] = (sample['close'] - sample['low'].rolling(1500).min())/(sample['high'].rolling(1500).max() - sample['low'].rolling(1500).min())
sample['rvolume_1500'] = (sample['volume'] - sample['volume'].rolling(1500).min())/(sample['volume'].rolling(1500).max() - sample['volume'].rolling(1500).min())

sample.dropna(inplace=True)

In [7]:
sample.tail()

,open,high,low,close,volume,ropen_7,rhigh_7,rlow_7,rclose_7,rvolume_7,...,ropen_168,rhigh_168,rlow_168,rclose_168,rvolume_168,ropen_1500,rhigh_1500,rlow_1500,rclose_1500,rvolume_1500
datetime,,,,,,,,,,,,,,,,,,,,,
2021-12-25 12:00:00,4051.74,4104.08,4049.84,4083.25,15628.7721,0.384670,1.000000,0.362332,0.755114,1.000000,...,0.752494,0.884332,0.747708,0.831864,0.249513,0.401709,0.440073,0.400317,0.424805,0.039886
2021-12-25 13:00:00,4083.25,4083.45,4057.39,4076.89,8990.6770,0.746409,0.748843,0.431580,0.668980,0.355843,...,0.831864,0.832368,0.766725,0.815844,0.113787,0.424805,0.424952,0.405851,0.420143,0.018190
2021-12-25 14:00:00,4076.90,4079.05,4060.00,4068.00,6758.6592,0.669102,0.695276,0.463355,0.560750,0.139249,...,0.815869,0.821285,0.773300,0.793451,0.068150,0.420151,0.421727,0.407764,0.413627,0.010894
2021-12-25 15:00:00,4068.00,4081.00,4061.34,4079.04,6177.4868,0.560750,0.719016,0.479669,0.695155,0.082853,...,0.793451,0.826196,0.776675,0.821259,0.056267,0.413627,0.423156,0.408746,0.421719,0.008995
2021-12-25 16:00:00,4079.04,4100.00,4079.03,4088.91,4139.8200,0.695155,0.950329,0.695033,0.815315,0.000000,...,0.821259,0.874055,0.821234,0.846121,0.014604,0.421719,0.437082,0.421712,0.428954,0.002334


In [48]:
# test cell for reward function
start_idx = random.randint(0, len(sample))
gamma = 1.
total_reward = 0.
stoc_reward = 0.
for idx in range(start_idx, start_idx+1000):
    this_idx = sample.index[idx]
    next_idx = sample.index[idx+1]
    this_close= sample.loc[this_idx, 'close']
    next_high = sample.loc[next_idx, 'high']
    next_low = sample.loc[next_idx, 'low']
    next_close = sample.loc[next_idx, 'close']
    reward = math.log(random.uniform(next_low, next_high)/this_close)
    p = random.random()
    total_reward += math.log(next_close/this_close)
    if p > 0.5: stoc_reward += reward
    else: continue
print(math.exp(total_reward))
print(math.exp(stoc_reward))

0.9867666968657464
0.9991722938943052
